# KubeCon India 2026 — Platform + App Setup (AWS / ACK track)

The notebook form of [`setup-with-ack.sh`](setup-with-ack.sh) — run **after**
[`00_init-with-ack.ipynb`](00_init-with-ack.ipynb). Applies the ACK backing of the
`bucket` claim and deploys the Application. Like ACK itself (no composition layer),
Step 1 is just one `vela def apply`.

## Steps
1. Apply the `bucket` backing (`bucket-ack.cue`)
2. Build + push the app images (product-catalog + bucket-browser)
3. Create AWS credentials in the app namespaces (dev / staging / prod)
4. Deploy the Application (`product-catalog.yaml`)
5. Verify


## Step 1: Apply the `bucket` backing (the How)

ACK has no XRD/Composition layer — Step 1 is a single `vela def apply` of the ACK
backing. (Apply exactly one of bucket-xp / bucket-ack / bucket-kcc.)

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
source "$REPO_ROOT/scripts/common.sh"

print_step "Applying the 'bucket' ComponentDefinition (ACK backing)"
vela def apply "$REPO_ROOT/platform/kubevela/components/bucket-ack.cue"
print_success "'bucket' ComponentDefinition (ACK) installed"


## Step 2: Build + push the app images

Build both images — the **product-catalog** API and the **bucket-browser** web UI (the
two `webservice` components). The apps are cloud-neutral; the same images run on AWS or
GCP.

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
bash "$REPO_ROOT/apps/product-catalog/build-image.sh"
bash "$REPO_ROOT/apps/bucket-browser/build-image.sh"


## Step 3: AWS credentials in the app namespaces

App pods mount an `aws-credentials` secret to reach S3 (the app's runtime creds —
separate from the ACK controller's). Load `.env.aws`, then create the secret per env.

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
source "$REPO_ROOT/scripts/common.sh"
source "$REPO_ROOT/scripts/load-aws-env.sh"
source "$REPO_ROOT/scripts/create-aws-secret.sh"

load_aws_env "$PWD/.env.aws"
for ns in dev staging prod; do
    create_aws_secret --create-namespace "$ns" aws-credentials
done


## Step 4: Deploy the Application

Deploy the SAME `product-catalog.yaml` as the Crossplane track — the `bucket` claim now
resolves via ACK because `bucket-ack.cue` is the installed backing. Multi-env workflow
auto-deploys dev, then suspends.

In [ ]:
%%bash
set -euo pipefail
DEMO_DIR="$(cd .. && pwd)"
source "$(cd ../../.. && pwd)/scripts/common.sh"

vela up -f "$DEMO_DIR/kubevela/product-catalog.yaml"
print_success "Application submitted (workflow deploys dev, then suspends for approval)"


## Step 5: Verify

The workflow's `request` steps already exercised the API. The bucket-browser UI lets you
see the objects.

In [ ]:
%%bash
set -euo pipefail
vela status product-catalog || true
echo
kubectl get pods,hpa -n dev
echo
echo "The ACK-provisioned S3 bucket(s):"
kubectl get buckets.s3.services.k8s.aws -A || true
echo
echo "Browse the objects (run in a separate terminal):"
echo "  kubectl -n dev port-forward svc/bucket-browser 8080:8080   # http://localhost:8080"


## Setup complete

The ACK track is deployed (API + bucket-browser + an ACK S3 bucket per env).

### Teardown
ACK has no force-destroy, so empty the buckets first, then delete the cluster:
```bash
./teardown-with-ack.sh   # empties the S3 buckets, then deletes the Application
../cleanup.sh            # deletes the cluster + registry
```
